# Упражнение: Позиционное кодирование и блок трансформера: Post-LN и Pre-LN

## Задание

В `input.csv` — эмбеддинги 4 токенов (`token, e0…e5`, d_model = 6). В `weights.json` — веса подслоя FFN
`W1` (6 × 12), `b1` (12), `W2` (12 × 6), `b2` (6) и параметры LayerNorm `gamma`, `beta` (по 6).

1. **Позиционное кодирование** (позиции pos = 0, 1, 2, 3; i = 0, 1, 2):
   PE(pos, 2i) = sin(pos / 10000^(2i/d)), PE(pos, 2i+1) = cos(pos / 10000^(2i/d)), d = 6.
   Вход блока: x = E + PE.
2. **FFN** (применяется к каждому токену отдельно): F(z) = ReLU(z·W1 + b1)·W2 + b2.
3. **LayerNorm** по признакам каждого токена: LN(z) = γ ⊙ (z − mean(z)) / √(var(z) + 10⁻⁵) + β,
   дисперсия — смещённая (делим на d, как `np.var`).
4. Два варианта остаточного блока с одним и тем же F и одной и той же LN:
   - **Post-LN** (исходная статья): post = LN(x + F(x));
   - **Pre-LN** (современные модели): pre = x + F(LN(x)).

## Подсказки

- Проверьте PE: в позиции 0 чётные измерения равны 0, нечётные — 1; в столбцах 0 и 1 частота самая высокая.
- Показатель степени 2i/d — по номеру **пары** измерений: для столбцов 0 и 1 он 0, для 2 и 3 — 2/6, для 4 и 5 — 4/6.
- `pe[:, 0::2] = np.sin(pos / 10000 ** (np.arange(0, d, 2) / d))`, аналогично cos для `1::2`;
  LayerNorm — `mean`/`var` по `axis=-1, keepdims=True`.

## Ожидаемый формат выходного файла

CSV с колонками `token, pos, dim, pe, post, pre` — 24 строки (токены в порядке файла, внутри dim = 0…5):
значение позиционного кодирования, выхода Post-LN и выхода Pre-LN блока. Числа округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/transformers
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.